# 地震 RMS 属性预测 gain：随机道 QC

本 notebook 读取 `wavelet_amplitude_gain_abs_vs_relative_attribute@20260429.ipynb` 的自适应 segment 样本，固定使用：

```text
target = gain_segment_ls
attribute = seismic_rms
model: ln(gain_segment_ls) = intercept + slope * ln(seismic_rms)
```

然后随机抽取若干条深度域地震道，计算同尺度的 RMS 属性曲线，并预测 `gain(z)` 的趋势。

注意：随机道没有井曲线与合成记录，因此这里不是拟合质量验证，只是检查属性驱动 gain trend 的空间应用形态是否合理。


In [ ]:
import json
import sys
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.seismic.survey import open_survey
from cup.utils.raw_trace import zscore_trace

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


## 1) 配置

In [ ]:
data_root = repo_root / "data"

segment_qc_dir = data_root / "output_wavelet_amplitude_gain_abs_vs_relative_attribute_20260429"
segment_samples_file = segment_qc_dir / "wavelet_amplitude_gain_abs_vs_relative_segment_samples.csv"

output_dir = data_root / "output_wavelet_amplitude_gain_attribute_apply_random_traces_20260429"
figure_dir = output_dir / "figures"
fit_metrics_file = output_dir / "wavelet_amplitude_gain_attribute_fit_metrics.csv"
random_trace_predictions_file = output_dir / "wavelet_amplitude_gain_attribute_random_trace_predictions.csv"
random_trace_summary_file = output_dir / "wavelet_amplitude_gain_attribute_random_trace_summary.csv"

seismic_file = data_root / "raw" / "mero_84_coord_extend"
segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}

random_seed = 20260429
random_trace_count = 8
max_random_attempts = 200

# Leave as None to infer the application RMS window from the median segment thickness.
application_rms_window_m = None
prediction_clip_percentiles = (5.0, 95.0)
attribute_floor_fraction = 0.10

required_paths = [segment_samples_file, seismic_file]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

print("Segment samples:", segment_samples_file)
print("Seismic:", seismic_file)
print("Output dir:", output_dir)


## 2) 工具函数

In [ ]:
def centered_moving_sum(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    window_samples = int(max(window_samples, 1))
    if values.size == 0:
        return values.copy()
    radius_left = window_samples // 2
    radius_right = window_samples - 1 - radius_left
    padded = np.pad(values, (radius_left, radius_right), mode="constant", constant_values=0.0)
    cumsum = np.concatenate([[0.0], np.cumsum(padded)])
    return cumsum[window_samples:] - cumsum[:-window_samples]


def centered_moving_rms(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    numerator = centered_moving_sum(np.where(valid, values**2, 0.0), window_samples)
    denominator = centered_moving_sum(valid.astype(float), window_samples)
    out = np.full(values.shape, np.nan, dtype=float)
    positive = denominator > 0.0
    out[positive] = np.sqrt(numerator[positive] / denominator[positive])
    return out


def resolve_odd_window_samples(window_m: float, sample_step_m: float) -> int:
    window_samples = int(round(float(window_m) / float(sample_step_m)))
    window_samples = max(window_samples, 3)
    if window_samples % 2 == 0:
        window_samples += 1
    return window_samples


def finite_pearson(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    if np.nanstd(x[mask]) <= 0.0 or np.nanstd(y[mask]) <= 0.0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])


def fit_log_linear(x_log: np.ndarray, y_log: np.ndarray) -> dict:
    x_log = np.asarray(x_log, dtype=float)
    y_log = np.asarray(y_log, dtype=float)
    mask = np.isfinite(x_log) & np.isfinite(y_log)
    if int(mask.sum()) < 5:
        raise ValueError("Need at least five finite samples for log-linear fitting.")
    x = x_log[mask]
    y = y_log[mask]
    design = np.column_stack([np.ones_like(x), x])
    intercept, slope = np.linalg.lstsq(design, y, rcond=None)[0]
    pred = intercept + slope * x
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 0.0 else 1.0 - ss_res / ss_tot
    return {
        "intercept": float(intercept),
        "slope": float(slope),
        "n_samples": int(mask.sum()),
        "pearson": finite_pearson(x, y),
        "r2_in_sample": float(r2),
        "mae_log_gain": float(np.mean(np.abs(y - pred))),
        "rmse_log_gain": float(np.sqrt(np.mean((y - pred) ** 2))),
    }


def predict_gain_from_rms(
    seismic_values: np.ndarray,
    *,
    window_samples: int,
    intercept: float,
    slope: float,
    attribute_floor: float,
    log_gain_clip: tuple[float, float],
) -> pd.DataFrame:
    seismic_raw = np.asarray(seismic_values, dtype=float)
    seismic_zscore = zscore_trace(seismic_raw)
    seismic_rms = centered_moving_rms(seismic_zscore, window_samples)
    rms_safe = np.maximum(seismic_rms, float(attribute_floor))
    log_seismic_rms = np.where(np.isfinite(rms_safe) & (rms_safe > 0.0), np.log(rms_safe), np.nan)
    log_gain_pred = float(intercept) + float(slope) * log_seismic_rms
    log_gain_pred_clipped = np.clip(log_gain_pred, float(log_gain_clip[0]), float(log_gain_clip[1]))
    return pd.DataFrame(
        {
            "seismic_raw": seismic_raw,
            "seismic_zscore": seismic_zscore,
            "seismic_rms": seismic_rms,
            "log_seismic_rms": log_seismic_rms,
            "log_gain_pred": log_gain_pred,
            "log_gain_pred_clipped": log_gain_pred_clipped,
            "gain_pred": np.exp(log_gain_pred),
            "gain_pred_clipped": np.exp(log_gain_pred_clipped),
        }
    )


## 3) 拟合 `ln(gain_segment_ls)` 与 `ln(seismic_rms)`

In [ ]:
segment_samples_df = pd.read_csv(segment_samples_file)
required_columns = {"log_gain_segment_ls", "log_seismic_rms", "gain_segment_ls", "seismic_rms", "tvdss_min_m", "tvdss_max_m"}
missing = required_columns - set(segment_samples_df.columns)
if missing:
    raise ValueError(f"Segment samples missing columns: {sorted(missing)}")

fit_df = segment_samples_df.loc[
    np.isfinite(segment_samples_df["log_gain_segment_ls"])
    & np.isfinite(segment_samples_df["log_seismic_rms"])
].copy()
if fit_df.empty:
    raise ValueError("No finite ln(gain_segment_ls) / ln(seismic_rms) samples.")

fit = fit_log_linear(
    fit_df["log_seismic_rms"].to_numpy(dtype=float),
    fit_df["log_gain_segment_ls"].to_numpy(dtype=float),
)
log_gain_clip = tuple(np.nanpercentile(fit_df["log_gain_segment_ls"].to_numpy(dtype=float), prediction_clip_percentiles))
attribute_floor = float(np.nanpercentile(fit_df["seismic_rms"].to_numpy(dtype=float), 1.0) * attribute_floor_fraction)
attribute_floor = max(attribute_floor, np.finfo(float).tiny)

segment_thickness_m = (fit_df["tvdss_max_m"] - fit_df["tvdss_min_m"]).to_numpy(dtype=float)
segment_thickness_m = segment_thickness_m[np.isfinite(segment_thickness_m) & (segment_thickness_m > 0.0)]
if application_rms_window_m is None:
    application_window_m = float(np.nanmedian(segment_thickness_m))
else:
    application_window_m = float(application_rms_window_m)

fit_metrics_df = pd.DataFrame(
    [
        {
            **fit,
            "target": "log_gain_segment_ls",
            "attribute": "log_seismic_rms",
            "log_gain_clip_p05": float(log_gain_clip[0]),
            "log_gain_clip_p95": float(log_gain_clip[1]),
            "gain_clip_p05": float(np.exp(log_gain_clip[0])),
            "gain_clip_p95": float(np.exp(log_gain_clip[1])),
            "attribute_floor": attribute_floor,
            "application_window_m": application_window_m,
            "random_seed": int(random_seed),
        }
    ]
)
fit_metrics_df.to_csv(fit_metrics_file, index=False)

print(f"ln(gain) = {fit['intercept']:.3f} + {fit['slope']:.3f} * ln(seismic_rms)")
print("n=", fit["n_samples"], "pearson=", f"{fit['pearson']:.3f}", "R2=", f"{fit['r2_in_sample']:.3f}")
print("gain clip:", float(np.exp(log_gain_clip[0])), "to", float(np.exp(log_gain_clip[1])))
print("application_window_m:", application_window_m)
print("Saved", fit_metrics_file)
display(fit_metrics_df)


## 4) 拟合关系 QC 图

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 5.2), constrained_layout=True)
well_names = fit_df["well_name"].drop_duplicates().tolist() if "well_name" in fit_df.columns else []
color_map = {well_name: plt.cm.tab10(idx % 10) for idx, well_name in enumerate(well_names)}  # type: ignore
if "well_name" in fit_df.columns:
    for well_name, well_df in fit_df.groupby("well_name"):
        ax.scatter(
            well_df["log_seismic_rms"],
            well_df["log_gain_segment_ls"],
            s=30,
            alpha=0.75,
            color=color_map[well_name],
            label=well_name,
        )
else:
    ax.scatter(fit_df["log_seismic_rms"], fit_df["log_gain_segment_ls"], s=30, alpha=0.75)

x_min, x_max = np.nanpercentile(fit_df["log_seismic_rms"], [1, 99])
x_line = np.linspace(float(x_min), float(x_max), 100)
y_line = fit["intercept"] + fit["slope"] * x_line
ax.plot(x_line, y_line, color="black", lw=1.4, label="OLS fit")
ax.set_xlabel("ln(seismic_rms)")
ax.set_ylabel("ln(gain_segment_ls)")
ax.set_title("Training segments")
ax.legend(loc="best", fontsize=7)
ax.grid(True, alpha=0.25)
fit_fig = figure_dir / "qc_01_gain_vs_seismic_rms_fit.png"
fig.savefig(fit_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", fit_fig)


## 5) 随机抽取地震道并预测 gain trend

In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options=segy_options,
)
geometry_depth = survey.query_geometry(domain="depth")
print(json.dumps(geometry_depth, indent=2, ensure_ascii=False))
assert geometry_depth["sample_domain"] == "depth"
assert geometry_depth["sample_unit"] == "m"

sample_step_m = float(geometry_depth["sample_step"])
window_samples = resolve_odd_window_samples(application_window_m, sample_step_m)
print("RMS window samples:", window_samples, "window_m:", window_samples * sample_step_m)

rng = np.random.default_rng(random_seed)
ilines = np.arange(
    float(geometry_depth["inline_min"]),
    float(geometry_depth["inline_max"]) + 0.5 * float(geometry_depth["inline_step"]),
    float(geometry_depth["inline_step"]),
)
xlines = np.arange(
    float(geometry_depth["xline_min"]),
    float(geometry_depth["xline_max"]) + 0.5 * float(geometry_depth["xline_step"]),
    float(geometry_depth["xline_step"]),
)

prediction_frames = []
summary_rows = []
attempts = 0
trace_id = 0
while trace_id < random_trace_count and attempts < max_random_attempts:
    attempts += 1
    inline = float(rng.choice(ilines))
    xline = float(rng.choice(xlines))
    try:
        x, y = survey.line_to_coord(inline, xline)
        seismic_trace = survey.import_seismic_at_well(x, y, domain="depth")
    except Exception as exc:
        continue

    depth_m = np.asarray(seismic_trace.basis, dtype=float)
    seismic_values = np.asarray(seismic_trace.values, dtype=float)
    pred_df = predict_gain_from_rms(
        seismic_values,
        window_samples=window_samples,
        intercept=fit["intercept"],
        slope=fit["slope"],
        attribute_floor=attribute_floor,
        log_gain_clip=log_gain_clip,
    )
    pred_df.insert(0, "trace_id", trace_id)
    pred_df.insert(1, "inline", inline)
    pred_df.insert(2, "xline", xline)
    pred_df.insert(3, "x", float(x))
    pred_df.insert(4, "y", float(y))
    pred_df.insert(5, "depth_m", depth_m)
    prediction_frames.append(pred_df)

    finite_gain = pred_df["gain_pred_clipped"].to_numpy(dtype=float)
    finite_gain = finite_gain[np.isfinite(finite_gain)]
    summary_rows.append(
        {
            "trace_id": trace_id,
            "inline": inline,
            "xline": xline,
            "x": float(x),
            "y": float(y),
            "n_samples": int(depth_m.size),
            "gain_pred_median": float(np.nanmedian(finite_gain)) if finite_gain.size else np.nan,
            "gain_pred_p10": float(np.nanpercentile(finite_gain, 10)) if finite_gain.size else np.nan,
            "gain_pred_p90": float(np.nanpercentile(finite_gain, 90)) if finite_gain.size else np.nan,
        }
    )
    trace_id += 1

if trace_id < random_trace_count:
    raise ValueError(f"Only extracted {trace_id}/{random_trace_count} random traces after {attempts} attempts.")

random_trace_predictions_df = pd.concat(prediction_frames, ignore_index=True)
random_trace_summary_df = pd.DataFrame(summary_rows)
random_trace_predictions_df.to_csv(random_trace_predictions_file, index=False)
random_trace_summary_df.to_csv(random_trace_summary_file, index=False)

print("Saved", random_trace_predictions_file)
print("Saved", random_trace_summary_file)
display(random_trace_summary_df)


## 6) 随机道结果图

In [ ]:
n_rows = int(np.ceil(random_trace_count / 2))
fig, axes = plt.subplots(n_rows, 2, figsize=(13.5, 3.2 * n_rows), squeeze=False, constrained_layout=True)
axes_flat = axes.ravel()

for ax, (trace_id, trace_df) in zip(axes_flat, random_trace_predictions_df.groupby("trace_id")):
    depth = trace_df["depth_m"].to_numpy(dtype=float)
    seismic_zscore = trace_df["seismic_zscore"].to_numpy(dtype=float)
    gain = trace_df["gain_pred_clipped"].to_numpy(dtype=float)
    rms = trace_df["seismic_rms"].to_numpy(dtype=float)

    seismic_scale = np.nanmax(np.abs(seismic_zscore))
    seismic_norm = seismic_zscore / seismic_scale if np.isfinite(seismic_scale) and seismic_scale > 0.0 else seismic_zscore
    gain_norm = gain / np.nanmedian(gain[np.isfinite(gain)]) if np.any(np.isfinite(gain)) else gain
    rms_norm = rms / np.nanmedian(rms[np.isfinite(rms)]) if np.any(np.isfinite(rms)) else rms

    ax.plot(seismic_norm, depth, color="0.15", lw=0.8, label="seismic zscore / maxabs")
    ax.plot(rms_norm, depth, color="tab:green", lw=1.0, alpha=0.85, label="RMS / median")
    ax.plot(gain_norm, depth, color="tab:blue", lw=1.0, alpha=0.9, label="gain / median")
    ax.invert_yaxis()
    ax.set_title(f"trace {trace_id}: IL {trace_df['inline'].iloc[0]:.0f}, XL {trace_df['xline'].iloc[0]:.0f}")
    ax.set_xlabel("normalized value")
    ax.set_ylabel("Depth (m)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=7)

for ax in axes_flat[random_trace_count:]:
    ax.axis("off")

random_fig = figure_dir / "qc_02_random_trace_predicted_gain.png"
fig.savefig(random_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", random_fig)


## 7) 输出清单

In [ ]:
print("Fit metrics:")
print(fit_metrics_file)
print("\nRandom trace predictions:")
print(random_trace_predictions_file)
print(random_trace_summary_file)
print("\nFigures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
